# Part 2 - CPU vs. GPU Computing in Python

This notebook is meant to be run **on the SCC**, on a node with a GPU, since it needs GPU-enabled libraries and hardware.

## Why GPUs?
A CPU has a small number of powerful cores, good at complex, sequential tasks. A GPU has thousands of simpler cores, designed to perform the **same operation on many pieces of data at once** - exactly the vectorized, array-based operations you've been using with NumPy. This makes GPUs extremely well suited to:

- Large matrix and tensor operations (deep learning, simulations)
- Operations on large regular grids of data (image processing, some materials simulations)
- Any workload dominated by highly parallel numerical computation

GPUs are *not* automatically faster for every task - for small data or highly sequential logic, the overhead of moving data to the GPU can outweigh the benefit.

## NumPy (CPU) vs. a GPU Array Library
NumPy arrays always live in the computer's regular memory (RAM) and computations run on the CPU. To run the same kind of array math on a GPU, we need a GPU-aware library. Two common choices:

- **CuPy**: a near drop-in replacement for NumPy that runs on NVIDIA GPUs (`import cupy as cp` instead of `import numpy as np`).
- **PyTorch** (`torch`): originally built for deep learning, but also a general-purpose GPU array library with a NumPy-like API.

On the SCC, load a GPU-enabled Python module and request a GPU node before running this notebook, e.g.:

```bash
module load python3/3.12.4
module load cuda/12.2
module load pytorch/2.3
```

(Exact module names/versions may differ - check `module avail` on the SCC.)

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("Is a GPU available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## Comparing CPU and GPU Performance
Let's do a large matrix multiplication on the CPU and on the GPU and compare timings. Matrix multiplication is a classic example of a highly parallel operation that benefits from a GPU.

In [ ]:
import time

size = 4000

# CPU tensors
a_cpu = torch.rand(size, size)
b_cpu = torch.rand(size, size)

start = time.time()
result_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start
print(f"CPU time: {cpu_time:.4f} seconds")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")

    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    torch.cuda.synchronize()  # Make sure any previous GPU work is done before timing
    start = time.time()
    result_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()  # Wait for the GPU computation to actually finish
    gpu_time = time.time() - start

    print(f"GPU time: {gpu_time:.4f} seconds")
    print(f"Speedup: {cpu_time / gpu_time:.1f}x")
else:
    print("No GPU available in this session - request a GPU node to run this cell.")

Notice the `torch.cuda.synchronize()` calls. GPU operations are launched **asynchronously** - the CPU hands off the work and moves on immediately. Without synchronizing, you'd measure only the (tiny) time it took to *launch* the operation, not the time it actually took to *run*.

## Moving Data Between CPU and GPU
Data must be explicitly moved to the GPU's memory before GPU operations can use it, and back to CPU memory before you can use it with regular NumPy/pandas/Matplotlib. This transfer has a cost, so for small amounts of data, staying on the CPU is often faster overall.

In [ ]:
import numpy as np

# NumPy array -> CPU tensor -> GPU tensor
stress_array = np.array([150.0, 200.0, 250.0, 300.0])
tensor_cpu = torch.from_numpy(stress_array)

if torch.cuda.is_available():
    tensor_gpu = tensor_cpu.to("cuda")
    print(tensor_gpu)

    # Bring it back to the CPU (and to NumPy) before further CPU-only processing
    back_to_cpu = tensor_gpu.cpu().numpy()
    print(back_to_cpu)

### *Exercise*
1. Change `size` in the matrix multiplication benchmark to something small (e.g., 50) and re-run the comparison. What happens to the speedup, and why?
2. Look up which SCC GPU models are available (`qgpus` or the SCC documentation). How much GPU memory does each have?

Next, we'll cover how to actually **request** a GPU node when submitting a job.